# Distillation Process for RadBERT for NER

To finetune our BERT model for domain-specific NER, we are using an OpenAI model (5.2) to match specific pydantic schemas. This could be done locally using the GoLLie model and framework for confidentiality.
We are starting with the 

### Create Schemas to Match to

In [1]:
from pydantic import BaseModel, Field, model_validator
from typing import Optional, Literal
from enum import Enum
from openai import OpenAI
import json

class RadiologyEntityLabel(str, Enum):
    ANATOMY = "ANATOMY"
    FINDING = "FINDING"
    FINDING_MODIFIER = "FINDING_MODIFIER"  # "mild", "severe", "stable"
    MEASUREMENT = "MEASUREMENT"
    MODALITY = "MODALITY"  # "CT", "MRI", "X-ray"
    LATERALITY = "LATERALITY"  # "left", "right", "bilateral"
    COMPARISON = "COMPARISON"  # "unchanged", "increased", "new"


class RadiologySpan(BaseModel):
    """A span annotation for radiology NER."""
    
    text: str = Field(description="Exact text as it appears")
    start: int = Field(description="Start character offset")
    end: int = Field(description="End character offset")
    label: RadiologyEntityLabel
    
    # Radiology-specific attributes
    is_negated: bool = Field(default=False, description="'No evidence of', 'without'")
    is_uncertain: bool = Field(default=False, description="'Possibly', 'cannot exclude'")
    clinical_significance: Optional[Literal["benign", "indeterminate", "suspicious", "critical"]] = None
    
    confidence: float = Field(default=1.0, ge=0.0, le=1.0)


class FindingAnatomyRelation(BaseModel):
    """Links a finding to its anatomical location."""
    finding_index: int
    anatomy_index: int


class FindingMeasurementRelation(BaseModel):
    """Links a finding to its measurement."""
    finding_index: int
    measurement_index: int


class RadiologyAnnotation(BaseModel):
    """Complete annotation for a radiology report."""
    
    text: str = Field(description="Source report text")
    entities: list[RadiologySpan] = Field(default_factory=list)
    
    # Relations
    finding_anatomy_relations: list[FindingAnatomyRelation] = Field(default_factory=list)
    finding_measurement_relations: list[FindingMeasurementRelation] = Field(default_factory=list)
    
    # Document-level
    modality: Optional[str] = None
    body_region: Optional[str] = None
    impression_severity: Optional[Literal["normal", "minor", "moderate", "severe", "critical"]] = None
    
    # Validate that the entity placement is correct
    @model_validator(mode='after')
    def validate_spans(self):
        for i, entity in enumerate(self.entities):
            extracted = self.text[entity.start:entity.end]
            if extracted.lower() != entity.text.lower():
                raise ValueError(
                    f"Entity {i} mismatch: '{entity.text}' vs '{extracted}' "
                    f"at [{entity.start}:{entity.end}]"
                )
        return self

### Load Data AND/OR Generate Synthetic Data

##### Load MedLane Data

In [2]:
from pathlib import Path
import sys

# Add repo root so `backend` is importable when running from notebooks/
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
import pandas as pd

# Load a single row from the merged plain language dataset
dataset_path = repo_root / "data" / "merged_plain_language_dataset.csv"
df = pd.read_csv(dataset_path)

medlane = df[df['source_dataset'] == 'MedLane']

print(f"Medical Report: {medlane.iloc[0]['original_text']}")
print(f"Plain Language Text: {medlane.iloc[0]['plain_language_text']}")

Medical Report: known lastname 51946 is a 31-year-old male s/p a a sibling matched allogeneic bone marrow transplant for severe aplastic anemia .
Plain Language Text: patient 51946 is a 31-year-old male after a a sibling matched allogeneic bone marrow transplant for severe aplastic lack of enough healthy red blood cells .


##### Generate Sythetic Medical Data Reports

In [ ]:
import asyncio
import json
from pydantic import BaseModel, Field
from typing import Optional, Literal
from enum import Enum
import os
from dotenv import load_dotenv
from typing import List, Dict, Optional, Type, TypeVar
from pydantic import BaseModel
import asyncio

from backend.src.utils.clients.llm_clients import AsyncOpenAIClient

class RadiologyReport(BaseModel):
    """A synthetic radiology report."""
    
    examination: str = Field(description="Type of exam (e.g., 'MR brain without and with contrast')")
    clinical_information: str = Field(description="Reason for exam (e.g., 'Chronic headache, dizziness')")
    comparison: str = Field(description="Prior studies compared (e.g., 'Brain MRI XX/XX/XXXX' or 'None')")
    technique: str = Field(description="How the exam was performed")
    findings: str = Field(description="Detailed findings section - the bulk of the report")
    impression: str = Field(description="Summary impression with numbered findings")
    
    modality: Literal["MRI", "CT", "X-ray", "Ultrasound"] = Field(description="Imaging modality")
    body_region: str = Field(description="Body part imaged")


class SyntheticReportBatch(BaseModel):
    """Batch of generated reports."""
    reports: list[RadiologyReport]

T = TypeVar('T', bound=BaseModel)

EXAMPLE_REPORT = """Examination(s): MR brain without and with contrast. Clinical information: Chronic headache, position. 
Comparison(s): Brain MRV with and without contrast XX/XX/XXXX, brain MRI and MRA without contrast XX/XX/XXXX, head CT without contrast XX/XX/XXXX. 
Technique: Multisequence, multiplanar MR of the brain obtained without and with intravenous contrast. 
Contrast type, dose and administration as documented in electronic medical record. 
Findings: Brain parenchyma: No diffusion restriction, abnormal susceptibility, enhancement. No mass. 
Few tiny foci of T2 prolongation in the supratentorial white matter, up to 3 mm. 
Midline: Superior convex margin to the anterior pituitary gland which is contained within the inferior half of the sella.
No flattened anterior pituitary or sellar remodeling typical for a partially empty/empty sella. Normal corpus callosum and pineal gland. 
No inferior cerebellar tonsillar herniation. 
Extra-axial: Mild expansion of the CSF signal/space at the left cerebellar medullary cistern could with apparent anterior displacement of the cisternal segment glossopharyngeal and vagal nerves, possible arachnoid cyst, up to 10 to 13 mm axial. No abnormal enhancement. Ventricles: Normal. Vessels: This exam not optimized to evaluate. Normal signal. Orbits: Slight increased anteroposterior dimension of the globes with posterior scleral thinning consistent with staphylomas. Paranasal sinuses and mastoids: Normal signal. 
Bones: No suspicious findings. 
Other soft tissues: No suspicious findings. 
Signal alteration consistent with a retention cyst in the posterior aspect of the nasopharynx to the left of midline measures up to 10 mm axial. 
Impression: 1. No acute intracranial abnormality. 2. Suspect a small arachnoid cyst at the left cerebellomedullary cistern, similar compared to prior exam from 2016. 
It probably displaces the cisternal segments 9th and 10th cranial nerves along its medial margin. Probably clinically benign."""

GENERATION_PROMPT = """You are an expert radiologist generating realistic synthetic radiology reports for medical AI training.

Generate radiology reports that match the style, structure, and medical terminology of real clinical reports. 

EXAMPLE REPORT FORMAT:
{example}

REQUIREMENTS:
1. Use realistic medical terminology and anatomical descriptions
2. Include specific measurements (mm, cm) where appropriate
3. Use hedging language appropriately ("possible", "suspected", "likely", "cannot exclude")
4. Include both normal and abnormal findings
5. Reference prior studies with XX/XX/XXXX date format
6. Impression should summarize key findings in numbered format
7. Vary the complexity - some normal studies, some with multiple findings
8. Include incidental findings occasionally
9. Use appropriate negation ("No evidence of", "Without", "Negative for")

MODALITIES AND BODY REGIONS TO COVER:
- MRI: Brain, Spine (cervical/thoracic/lumbar), Knee, Shoulder, Abdomen/Pelvis
- CT: Head, Chest, Abdomen/Pelvis, Spine
- X-ray: Chest, Extremities, Spine
- Ultrasound: Abdomen, Thyroid, Carotid

CLINICAL SCENARIOS TO INCLUDE:
- Headache, dizziness, vision changes
- Back pain, radiculopathy
- Trauma evaluation
- Cancer screening/staging
- Follow-up of known findings
- Shortness of breath, chest pain
- Abdominal pain
- Joint pain, injury

FINDING TYPES TO VARY:
- Completely normal studies
- Minor/incidental findings (cysts, small nodules)
- Moderate findings requiring follow-up
- Significant findings requiring urgent attention
- Stable findings compared to prior
- New or changed findings

Generate diverse, realistic reports that would challenge a medical text simplification system."""


async def generate_reports(
    client: AsyncOpenAIClient,
    num_reports: int = 5,
    modality: Optional[str] = None,
    body_region: Optional[str] = None,
    finding_severity: Optional[str] = None,
) -> list[RadiologyReport]:
    """Generate a batch of synthetic radiology reports."""
    
    specific_instructions = ""
    if modality:
        specific_instructions += f"\nModality: {modality}"
    if body_region:
        specific_instructions += f"\nBody Region: {body_region}"
    if finding_severity:
        specific_instructions += f"\nFinding Severity: {finding_severity}"
    
    user_prompt = f"Generate {num_reports} diverse, realistic radiology reports."
    if specific_instructions:
        user_prompt += f"\n\nSpecific requirements:{specific_instructions}"
    
    result = await client.generate_parsed(
        messages=[
            {"role": "system", "content": GENERATION_PROMPT.format(example=EXAMPLE_REPORT)},
            {"role": "user", "content": user_prompt}
        ],
        schema=SyntheticReportBatch
    )
    
    return result.reports


async def generate_diverse_report_set(
    client: AsyncOpenAIClient,
    total_reports: int = 20,
    batch_size: int = 5,
) -> list[RadiologyReport]:
    """Generate a diverse set of reports across modalities and findings."""
    
    # Define batches with specific characteristics for diversity
    batch_specs = [
        {"modality": "MRI", "body_region": "Brain", "finding_severity": "mixed"},
        {"modality": "CT", "body_region": "Chest", "finding_severity": "mixed"},
        {"modality": "MRI", "body_region": "Spine", "finding_severity": "mixed"},
        {"modality": "CT", "body_region": "Abdomen/Pelvis", "finding_severity": "mixed"},
    ]
    
    all_reports = []
    reports_per_batch = total_reports // len(batch_specs)
    
    tasks = []
    for spec in batch_specs:
        task = generate_reports(
            client=client,
            num_reports=reports_per_batch,
            modality=spec["modality"],
            body_region=spec["body_region"],
            finding_severity=spec["finding_severity"],
        )
        tasks.append(task)
    
    results = await asyncio.gather(*tasks)
    
    for batch in results:
        all_reports.extend(batch)
    
    print(f"Generated {len(all_reports)} reports")
    return all_reports


def format_report_as_text(report: RadiologyReport) -> str:
    """Format a RadiologyReport into a single text block like the example."""
    return (
        f"Examination(s): {report.examination}. "
        f"Clinical information: {report.clinical_information}. "
        f"Comparison(s): {report.comparison}. "
        f"Technique: {report.technique}. "
        f"Findings: {report.findings} "
        f"Impression: {report.impression}"
    )

In [23]:
import pandas as pd
from pathlib import Path
import json


client = AsyncOpenAIClient()

print("Generating 20 synthetic radiology reports...")
reports = await generate_diverse_report_set(client, total_reports=20, batch_size=5)

# Convert to DataFrame with full_text column
reports_data = []
for i, report in enumerate(reports):
    data = report.model_dump()
    data["report_id"] = i + 1
    data["full_text"] = format_report_as_text(report)
    reports_data.append(data)

df = pd.DataFrame(reports_data)

# Reorder columns
column_order = [
    "report_id",
    "modality", 
    "body_region",
    "examination",
    "clinical_information",
    "comparison",
    "technique",
    "findings",
    "impression",
    "full_text"
]
df = df[column_order]

# Save directly to CSV
csv_path = Path("synthetic_radiology_reports.csv")
df.to_csv(csv_path, index=False)
print(f"Saved {len(df)} reports to {csv_path}")

Generating 20 synthetic radiology reports...
Generated 20 reports
Saved 20 reports to synthetic_radiology_reports.csv


### Generate LLM Annotations

In [24]:
from backend.src.utils.clients.llm_clients import AsyncOpenAIClient

from typing import List, Dict, Optional, Type, TypeVar
from pydantic import BaseModel
import asyncio



T = TypeVar('T', bound=BaseModel)

RADIOLOGY_PROMPT = """You are annotating radiology reports for NER model training.
Extract entities with EXACT character offsets.

ENTITY LABELS:
- ANATOMY: Body parts, organs, structures (lung, liver, vertebra L4-L5, right kidney)
- FINDING: Observations/abnormalities (nodule, opacity, fracture, effusion, mass)
- FINDING_MODIFIER: Descriptors of findings (mild, moderate, severe, stable, chronic, acute)
- MEASUREMENT: Sizes and quantities (2.3 cm, 5 mm, 12 x 8 mm)
- MODALITY: Imaging type if mentioned (CT, MRI, ultrasound, radiograph)
- LATERALITY: Side indicators (left, right, bilateral, midline)
- COMPARISON: Change from prior (unchanged, increased, decreased, new, resolved, stable)

ATTRIBUTES:
- is_negated: True for "no", "without", "negative for", "no evidence of"
- is_uncertain: True for "possibly", "may represent", "cannot exclude", "questionable"
- clinical_significance: benign/indeterminate/suspicious/critical

RELATIONS:
- Link findings to their anatomy (nodule -> right upper lobe)
- Link findings to measurements (nodule -> 8mm)

EXAMPLE:
Text: "There is a 8 mm nodule in the right upper lobe, unchanged from prior. No pleural effusion."

Entities:
- "8 mm" [12:16] MEASUREMENT
- "nodule" [17:23] FINDING clinical_significance=indeterminate
- "right" [31:36] LATERALITY  
- "upper lobe" [37:47] ANATOMY
- "unchanged" [49:58] COMPARISON
- "No" [70:72] - (don't tag, it's a cue)
- "pleural effusion" [73:89] FINDING is_negated=true clinical_significance=benign

Relations:
- finding_anatomy: nodule (1) -> upper lobe (3)
- finding_measurement: nodule (1) -> 8 mm (0)
"""


async def generate_radiology_annotations_async(
    reports: list[str],
    client: AsyncOpenAIClient,
    schema: Type[T],
    batch_size: int = 10,
) -> tuple[list[T], list[tuple[int, str, str]]]:
    """
    Process multiple reports concurrently.
    
    Args:
        reports: List of radiology report texts
        client: Async OpenAI client instance
        schema: Pydantic model class for the annotation format
        batch_size: Number of concurrent requests per batch
        
    Returns:
        Tuple of (successful_annotations, failed_reports)
    """
    
    annotations = []
    failed = []
    
    async def process_one(idx: int, report: str) -> tuple[int, Optional[T], Optional[str]]:
        """Process a single report."""
        try:
            annotation = await client.generate_parsed(
                messages=[
                    {"role": "system", "content": RADIOLOGY_PROMPT},
                    {"role": "user", "content": f"Annotate this report:\n\n{report}"}
                ],
                schema=schema
            )
            # Set source text if the schema has a text field
            if hasattr(annotation, 'text'):
                annotation.text = report
            return idx, annotation, None
        except Exception as e:
            return idx, None, str(e)
    
    # Process in batches
    for batch_start in range(0, len(reports), batch_size):
        batch_end = min(batch_start + batch_size, len(reports))
        batch = reports[batch_start:batch_end]
        
        # Create tasks for this batch
        tasks = [
            process_one(batch_start + i, report) 
            for i, report in enumerate(batch)
        ]
        
        # Run batch concurrently
        results = await asyncio.gather(*tasks)
        
        # Collect results
        for idx, annotation, error in results:
            if annotation is not None:
                annotations.append(annotation)
            else:
                failed.append((idx, reports[idx], error))
        
        print(f"Processed {batch_end}/{len(reports)}")
    
    print(f"Successfully annotated {len(annotations)}/{len(reports)}")
    return annotations, failed

In [5]:
sample_reports = medlane['original_text'].head(10).tolist()
client = AsyncOpenAIClient()

annotations, failed = await generate_radiology_annotations_async(sample_reports, client, RadiologyAnnotation)

Processed 10/10
Successfully annotated 8/10


In [8]:
annotations

[RadiologyAnnotation(text='besides her episode of hypotension ( see above ) she did require her antihypertensive medications and her blood pressure was reasonably controlled .', entities=[RadiologySpan(text='hypotension', start=23, end=34, label=<RadiologyEntityLabel.FINDING: 'FINDING'>, is_negated=False, is_uncertain=False, clinical_significance='critical', confidence=0.9), RadiologySpan(text='blood pressure', start=106, end=120, label=<RadiologyEntityLabel.ANATOMY: 'ANATOMY'>, is_negated=False, is_uncertain=False, clinical_significance=None, confidence=0.95), RadiologySpan(text='reasonably controlled', start=125, end=146, label=<RadiologyEntityLabel.FINDING_MODIFIER: 'FINDING_MODIFIER'>, is_negated=False, is_uncertain=False, clinical_significance=None, confidence=0.9)], finding_anatomy_relations=[FindingAnatomyRelation(finding_index=0, anatomy_index=1)], finding_measurement_relations=[], modality=None, body_region=None, impression_severity=None),
 RadiologyAnnotation(text='her condit

In [ ]:
def load_reports_for_annotation(csv_path: str = "synthetic_radiology_reports.csv") -> list[str]:
    """Load reports from CSV and return list of full_text strings for annotation."""
    df = pd.read_csv(csv_path)
    return df["full_text"].tolist()


def load_reports_dataframe(csv_path: str = "synthetic_radiology_reports.csv") -> pd.DataFrame:
    """Load full DataFrame for analysis."""
    return pd.read_csv(csv_path)

reports_text = load_reports_for_annotation("synthetic_radiology_reports.csv")

annotations, failed = await generate_radiology_annotations_async(
    reports=reports_text,
    client=client,
    schema=RadiologyAnnotation,
    batch_size=5
)

In [ ]:
annotations

Processed 10/37000
Processed 20/37000


: 

### Create dataset for training

In [ ]:
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset

class RadiologyNERDataset(Dataset):
    """Dataset for fine-tuning RadBERT on radiology NER."""
    
    def __init__(
        self, 
        annotations: list[RadiologyAnnotation],
        tokenizer,
        label2id: dict,
        max_length: int = 512
    ):
        self.annotations = annotations
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        annotation = self.annotations[idx]
        
        # Tokenize with offset mapping
        encoding = self.tokenizer(
            annotation.text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_offsets_mapping=True,
            return_tensors='pt'
        )
        
        # Initialize labels
        labels = torch.full(
            (self.max_length,), 
            self.label2id['O'],
            dtype=torch.long
        )
        
        # Attribute labels for multi-task
        negation_labels = torch.zeros(self.max_length, dtype=torch.long)
        uncertainty_labels = torch.zeros(self.max_length, dtype=torch.long)
        
        # Map entities to tokens
        offset_mapping = encoding['offset_mapping'][0].tolist()
        
        for entity in annotation.entities:
            entity_start = entity.start
            entity_end = entity.end
            first_token = True
            
            for tok_idx, (tok_start, tok_end) in enumerate(offset_mapping):
                # Skip special tokens
                if tok_start == tok_end == 0:
                    continue
                
                # Check if token overlaps with entity
                if tok_start >= entity_start and tok_end <= entity_end:
                    if first_token:
                        labels[tok_idx] = self.label2id[f"B-{entity.label.value}"]
                        first_token = False
                    else:
                        labels[tok_idx] = self.label2id[f"I-{entity.label.value}"]
                    
                    # Set attribute labels
                    if entity.is_negated:
                        negation_labels[tok_idx] = 1
                    if entity.is_uncertain:
                        uncertainty_labels[tok_idx] = 1
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': labels,
            'negation_labels': negation_labels,
            'uncertainty_labels': uncertainty_labels,
        }


def create_label_mappings():
    """Create BIO label mappings for radiology NER."""
    
    labels = ['O']
    for entity_type in RadiologyEntityLabel:
        labels.append(f"B-{entity_type.value}")
        labels.append(f"I-{entity_type.value}")
    
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}
    
    return label2id, id2label

### Create Model Outline

In [ ]:
from transformers import AutoModel, PreTrainedModel, AutoConfig
import torch.nn as nn

class RadBERTForRadiologyNER(PreTrainedModel):
    """RadBERT with NER + negation + uncertainty heads."""
    
    def __init__(self, config, num_ner_labels):
        super().__init__(config)
        
        self.roberta = AutoModel.from_config(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        
        hidden_size = config.hidden_size
        
        # NER head
        self.ner_classifier = nn.Linear(hidden_size, num_ner_labels)
        
        # Attribute heads
        self.negation_classifier = nn.Linear(hidden_size, 2)
        self.uncertainty_classifier = nn.Linear(hidden_size, 2)
        
        # Loss weights
        self.ner_weight = 1.0
        self.negation_weight = 0.3
        self.uncertainty_weight = 0.3
        
    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None,
        negation_labels=None,
        uncertainty_labels=None,
    ):
        outputs = self.roberta(
            input_ids,
            attention_mask=attention_mask
        )
        
        sequence_output = self.dropout(outputs.last_hidden_state)
        
        # Get predictions from each head
        ner_logits = self.ner_classifier(sequence_output)
        negation_logits = self.negation_classifier(sequence_output)
        uncertainty_logits = self.uncertainty_classifier(sequence_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            
            # NER loss
            loss = self.ner_weight * loss_fct(
                ner_logits.view(-1, ner_logits.shape[-1]),
                labels.view(-1)
            )
            
            # Negation loss (only on entity tokens)
            if negation_labels is not None:
                entity_mask = labels != self.config.pad_token_id
                if entity_mask.any():
                    loss += self.negation_weight * loss_fct(
                        negation_logits.view(-1, 2),
                        negation_labels.view(-1)
                    )
            
            # Uncertainty loss
            if uncertainty_labels is not None:
                loss += self.uncertainty_weight * loss_fct(
                    uncertainty_logits.view(-1, 2),
                    uncertainty_labels.view(-1)
                )
        
        return {
            'loss': loss,
            'ner_logits': ner_logits,
            'negation_logits': negation_logits,
            'uncertainty_logits': uncertainty_logits,
        }

### Train and Evaluate Model

In [ ]:
from transformers import Trainer, TrainingArguments, AutoTokenizer
from sklearn.model_selection import train_test_split
import numpy as np


def compute_metrics(eval_pred):
    """Compute NER metrics."""
    predictions, labels = eval_pred
    
    # Get NER predictions (first output)
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    predictions = np.argmax(predictions, axis=2)
    
    # Flatten and filter padding
    true_labels = []
    pred_labels = []
    
    for pred_seq, label_seq in zip(predictions, labels):
        for pred, label in zip(pred_seq, label_seq):
            if label != -100:  # Ignore padding
                true_labels.append(label)
                pred_labels.append(pred)
    
    # Calculate metrics
    from sklearn.metrics import precision_recall_fscore_support
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, 
        pred_labels, 
        average='weighted',
        zero_division=0
    )
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


def train_radbert_ner(
    annotations: list[RadiologyAnnotation],
    output_dir: str = "./radbert-radiology-ner",
    model_name: str = "UCSD-VA-health/RadBERT-RoBERTa-4m",
    epochs: int = 5,
    batch_size: int = 16,
    learning_rate: float = 2e-5,
):
    """Train RadBERT on radiology NER data."""
    
    # Setup
    label2id, id2label = create_label_mappings()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Split data
    train_annotations, val_annotations = train_test_split(
        annotations, 
        test_size=0.1, 
        random_state=42
    )
    
    # Create datasets
    train_dataset = RadiologyNERDataset(train_annotations, tokenizer, label2id)
    val_dataset = RadiologyNERDataset(val_annotations, tokenizer, label2id)
    
    # Load model
    config = AutoConfig.from_pretrained(model_name)
    config.num_labels = len(label2id)
    config.id2label = id2label
    config.label2id = label2id
    
    model = RadBERTForRadiologyNER.from_pretrained(
        model_name,
        config=config,
        num_ner_labels=len(label2id),
        ignore_mismatched_sizes=True
    )
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=50,
        warmup_ratio=0.1,
        fp16=True,
    )
    
    # Custom data collator
    def collate_fn(batch):
        return {
            'input_ids': torch.stack([x['input_ids'] for x in batch]),
            'attention_mask': torch.stack([x['attention_mask'] for x in batch]),
            'labels': torch.stack([x['labels'] for x in batch]),
            'negation_labels': torch.stack([x['negation_labels'] for x in batch]),
            'uncertainty_labels': torch.stack([x['uncertainty_labels'] for x in batch]),
        }
    
    # Train
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        data_collator=collate_fn,
    )
    
    trainer.train()
    
    # Save
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    # Save label mappings
    with open(f"{output_dir}/label_mappings.json", "w") as f:
        json.dump({"label2id": label2id, "id2label": id2label}, f)
    
    return model, tokenizer

### Create Inference Pipeline

In [ ]:
class RadiologyNERPipeline:
    """Production inference pipeline for radiology NER."""
    
    def __init__(self, model_path: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        with open(f"{model_path}/label_mappings.json") as f:
            mappings = json.load(f)
        
        self.label2id = mappings['label2id']
        self.id2label = {int(k): v for k, v in mappings['id2label'].items()}
        
        config = AutoConfig.from_pretrained(model_path)
        self.model = RadBERTForRadiologyNER.from_pretrained(
            model_path,
            config=config,
            num_ner_labels=len(self.label2id)
        )
        self.model.eval()
        
        if torch.cuda.is_available():
            self.model = self.model.cuda()
    
    def predict(self, text: str) -> list[dict]:
        """Extract entities from a radiology report."""
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            return_tensors='pt',
            return_offsets_mapping=True,
            truncation=True,
            max_length=512
        )
        
        offset_mapping = encoding.pop('offset_mapping')[0].tolist()
        
        if torch.cuda.is_available():
            encoding = {k: v.cuda() for k, v in encoding.items()}
        
        # Predict
        with torch.no_grad():
            outputs = self.model(**encoding)
        
        ner_preds = torch.argmax(outputs['ner_logits'], dim=-1)[0].cpu().tolist()
        neg_preds = torch.argmax(outputs['negation_logits'], dim=-1)[0].cpu().tolist()
        unc_preds = torch.argmax(outputs['uncertainty_logits'], dim=-1)[0].cpu().tolist()
        
        # Decode entities
        entities = []
        current_entity = None
        
        for idx, (pred, (start, end)) in enumerate(zip(ner_preds, offset_mapping)):
            if start == end == 0:  # Special token
                continue
            
            label = self.id2label[pred]
            
            if label.startswith('B-'):
                # Save previous entity
                if current_entity:
                    entities.append(current_entity)
                
                # Start new entity
                entity_type = label[2:]
                current_entity = {
                    'text': text[start:end],
                    'start': start,
                    'end': end,
                    'label': entity_type,
                    'is_negated': bool(neg_preds[idx]),
                    'is_uncertain': bool(unc_preds[idx]),
                }
            
            elif label.startswith('I-') and current_entity:
                # Continue entity
                current_entity['text'] = text[current_entity['start']:end]
                current_entity['end'] = end
            
            else:
                # Outside - save any current entity
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None
        
        # Don't forget last entity
        if current_entity:
            entities.append(current_entity)
        
        return entities
    
    def batch_predict(self, texts: list[str], batch_size: int = 32) -> list[list[dict]]:
        """Batch inference for efficiency."""
        
        all_entities = []
        
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            
            for text in batch_texts:
                entities = self.predict(text)
                all_entities.append(entities)
        
        return all_entities

### Test Output

In [ ]:
if __name__ == "__main__":
    # Sample radiology reports (in practice, load your dataset)
    sample_reports = [
        """FINDINGS: The lungs are clear. Heart size is normal. There is no pleural 
        effusion or pneumothorax. The mediastinal contours are unremarkable.
        IMPRESSION: Normal chest radiograph.""",
        
        """FINDINGS: There is a 1.2 cm nodule in the right upper lobe, new compared 
        to prior study from 6 months ago. Mild dependent atelectasis at the lung bases. 
        No pleural effusion. Heart size is normal.
        IMPRESSION: New right upper lobe nodule, recommend follow-up CT in 3 months.""",
        
        """FINDINGS: Redemonstration of known 8mm left lower lobe nodule, unchanged. 
        Possible ground glass opacity in the right middle lobe, may represent early 
        infection versus inflammation. No lymphadenopathy. Small pericardial effusion.
        IMPRESSION: 1. Stable left lower lobe nodule. 2. Right middle lobe ground 
        glass opacity - correlate clinically.""",
    ]
    
    # Initialize OpenAI client
    client = OpenAI()
    
    # Step 1: Generate training data
    print("Generating training annotations...")
    annotations, failed = generate_radiology_annotations(sample_reports, client)
    
    # Step 2: Inspect an annotation
    print("\nSample annotation:")
    print(json.dumps(annotations[0].model_dump(), indent=2, default=str))
    
    # Step 3: Train model (need more data in practice - at least 500-1000)
    # print("\nTraining RadBERT...")
    # model, tokenizer = train_radbert_ner(annotations)
    
    # Step 4: Inference
    # pipeline = RadiologyNERPipeline("./radbert-radiology-ner")
    # 
    # test_report = "There is a 2.5 cm mass in the left kidney, suspicious for malignancy."
    # entities = pipeline.predict(test_report)
    # 
    # for entity in entities:
    #     neg = "[NEG]" if entity['is_negated'] else ""
    #     unc = "[UNC]" if entity['is_uncertain'] else ""
    #     print(f"  {entity['text']:20} | {entity['label']:15} {neg} {unc}")


In [ ]:
# Save synthetic reports to CSV (run after generation)
import pandas as pd
import json
from pathlib import Path

json_path = Path("synthetic_radiology_reports.json")
if json_path.exists():
    reports_json = json.loads(json_path.read_text())
    pd.DataFrame(reports_json).to_csv("synthetic_radiology_reports.csv", index=False)
    print("Saved to synthetic_radiology_reports.csv")
else:
    raise FileNotFoundError("synthetic_radiology_reports.json not found. Run the generation cell first.")